In [ ]:
!pip install -q huggingface_hub pandas

In [ ]:
from huggingface_hub import InferenceClient
import pandas as pd
import time

HF_TOKEN = "hf_xxxxxxxxxxxxxxxxx"

QUESTION = """
Xin luật sư cho tôi hỏi:
mua lại công ty đang hoạt động thì người nhận chuyển nhượng cần kiểm tra rủi ro thuế gì?
"""

MODELS = [
{
"name": "CMC-AI-Legal-32B",
"id": "CMC-OPENAI/CMC-AI-Legal-32B"
},
{
"name": "ChatLaw-13B",
"id": "pandalla/ChatLaw-13B"
},
{
"name": "LawGPT-7B",
"id": "lawgpt/LawGPT-7B-beta1.0"
},
{
"name": "Lawyer-LLaMA-13B",
"id": "AndrewZhe/lawyer-llama-13b"
},
{
"name": "SaulLM-7B",
"id": "Equall/Saul-7B-Instruct-v1"
},
{
"name": "KL3M",
"id": "alea-institute/kl3m-3b"
}
]

client = InferenceClient(api_key=HF_TOKEN)

In [ ]:
results = []

for model in MODELS:

print("=" * 80)
print("MODEL:", model["name"])

answer = ""
status = ""

try:

    try:
        response = client.chat_completion(
            model=model["id"],
            messages=[
                {
                    "role": "user",
                    "content": QUESTION
                }
            ],
            max_tokens=512
        )

        answer = response.choices[0].message.content
        status = "SUCCESS"

    except Exception:

        answer = client.text_generation(
            QUESTION,
            model=model["id"],
            max_new_tokens=512
        )

        status = "SUCCESS"

except Exception as e:

    status = f"FAILED: {str(e)}"
    answer = ""

results.append({
    "Model": model["name"],
    "Model_ID": model["id"],
    "Question": QUESTION.strip(),
    "Answer": answer,
    "Status": status
})

print(status)
time.sleep(2)

df = pd.DataFrame(results)

df.to_csv(
"results.csv",
index=False,
encoding="utf-8-sig"
)

print("\nSaved: results.csv")
display(df)